In [ ]:
# Databricks notebook source



# RAG Pipeline Setup

This notebook sets up the RAG pipeline by:
1. Loading complaint data
2. Chunking documents
3. Generating embeddings
4. Storing everything in Delta tables for reuse

Run this notebook once to set up the pipeline, then use the summarization notebook to generate summaries.



## Configuration



In [ ]:

import os
import pandas as pd
import numpy as np
import json
import requests
import re
from typing import Dict, List, Tuple, Optional
from datetime import datetime

# Configuration
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
EMBEDDING_ENDPOINT = "gte-endpoint"
WORKSPACE_URL = "https://adb-7941446833400015.15.azuredatabricks.net"

# Delta table locations
CATALOG = "cntrl-busops-dev"
SCHEMA = "complaints-1kh-gld"
SOURCE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`vw_complaint`"
CHUNKS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`complaint_chunks`"
METADATA_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`rag_pipeline_metadata`"

# Vector Search Setup
VECTOR_SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.complaints_chunk_index"

print("Configuration loaded")
print(f"Source table: {SOURCE_TABLE}")
print(f"Chunks will be saved to: {CHUNKS_TABLE}")
print(f"Metadata will be saved to: {METADATA_TABLE}")



## Load Source Data



In [ ]:

print("Loading complaint data...")
df_spark = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")
df = df_spark.toPandas()

print(f"Loaded {len(df)} complaints")



## Extract Reference Numbers



In [ ]:

def extract_reference_number(text: str) -> str:
    """Extract reference_number from the all_columns text field."""
    if pd.isna(text) or not text:
        return "unknown"
    
    match = re.search(r'reference_number:\s*([^\|]+)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    id_match = re.search(r'(CDCR\w[|C-\w]{5,})', text)
    if id_match:
        return id_match.group(1)
    
    return "unknown"

# Extract reference numbers
df['reference_number'] = df['all_columns'].apply(extract_reference_number)
all_ids = df['reference_number'].unique().tolist()
docs = df['all_columns'].fillna("(no content)").tolist()

print(f"Found {len(all_ids)} unique complaints")



## Document Chunking



In [ ]:

def _chunk_text_with_overlap(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Split text into overlapping chunks."""
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        
        if end < len(text):
            search_start = end - int(chunk_size * 0.2)
            search_region = text[search_start:end]
            last_period = search_region.rfind('. ')
            last_newline = search_region.rfind('\n')
            last_pipe = search_region.rfind(' | ')
            last_break = max(last_period, last_newline, last_pipe)
            
            if last_break != -1:
                end = search_start + last_break + 1
        
        chunks.append(text[start:end].strip())
        start = end - overlap
    
    return chunks


def smart_chunk_document(
    doc: str,
    chunk_size: int = 512,
    chunk_overlap: int = 128,
    preserve_sections: bool = True
) -> List[Tuple[str, Dict[str, str]]]:
    """Smart chunking with section preservation."""
    
    chunks = []
    reference_number = extract_reference_number(doc)
    
    if preserve_sections:
        sections = re.split(r'(^##\s+\$)', doc, flags=re.MULTILINE)
        current_section = ""
        section_name = "Main"
        
        for i, part in enumerate(sections):
            if part.strip().startswith("## "):
                section_name = part.strip().replace("## ", "")
                current_section = part + "\n"
            elif part.strip():
                current_section += part
                
                if len(current_section) >= chunk_size:
                    section_chunks = _chunk_text_with_overlap(current_section, chunk_size, chunk_overlap)
                    
                    for j, chunk_text in enumerate(section_chunks):
                        metadata = {
                            "reference_number": reference_number,
                            "section": section_name,
                            "chunk_index": j,
                            "total_chunks": len(section_chunks)
                        }
                        chunks.append((chunk_text, metadata))
                    current_section = ""
        
        if current_section.strip():
            metadata = {
                "reference_number": reference_number,
                "section": section_name,
                "chunk_index": 0,
                "total_chunks": 1
            }
            chunks.append((current_section, metadata))
    else:
        chunk_texts_list = _chunk_text_with_overlap(doc, chunk_size, chunk_overlap)
        for i, chunk_text in enumerate(chunk_texts_list):
            metadata = {
                "reference_number": reference_number,
                "section": "Full-doc",
                "chunk_index": i,
                "total_chunks": len(chunk_texts_list)
            }
            chunks.append((chunk_text, metadata))
    
    if not chunks and doc.strip():
        metadata = {
            "reference_number": reference_number,
            "section": "Main",
            "chunk_index": 0,
            "total_chunks": 1
        }
        chunks.append((doc, metadata))
    
    return chunks

# Build chunks
print("Chunking documents...")
chunk_texts = []
chunk_metadata = []

for i, doc in enumerate(docs):
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(docs)} documents...")
    
    chunks = smart_chunk_document(doc, chunk_size=512, chunk_overlap=128, preserve_sections=True)
    for chunk_text, metadata in chunks:
        chunk_texts.append(chunk_text)
        chunk_metadata.append(metadata)

print(f"Created {len(chunk_texts)} chunks from {len(docs)} complaints")
print(f"Average chunks per complaint: {len(chunk_texts) / len(docs):.1f}")



## Generate Embeddings



In [ ]:

def _invocations_url(workspace_url: str, endpoint_name: str) -> str:
    return f"{workspace_url.rstrip('/')}/serving-endpoints/{endpoint_name}/invocations"


def _make_headers(token: str) -> dict:
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


def _parse_embeddings(resp_json):
    """Handle common embedding response shapes."""
    if isinstance(resp_json, dict):
        if "embeddings" in resp_json and isinstance(resp_json["embeddings"], list):
            return resp_json["embeddings"]
        if "data" in resp_json:
            data = resp_json["data"]
            if isinstance(data, list) and data and isinstance(data[0], dict) and "embedding" in data[0]:
                return [row["embedding"] for row in data]
            if isinstance(data, dict) and "embeddings" in data:
                return data["embeddings"]
    raise ValueError(f"Unrecognized embeddings response shape")


def embed_databricks(
    texts: List[str],
    workspace_url: str = WORKSPACE_URL,
    endpoint_name: str = EMBEDDING_ENDPOINT,
    token: str = DATABRICKS_TOKEN,
    batch_size: int = 64,
) -> np.ndarray:
    """Returns L2-normalized embeddings."""
    url = _invocations_url(workspace_url, endpoint_name)
    headers = _make_headers(token)
    
    all_vecs: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        payload = {"input": batch}
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        if response.ok:
            vecs = _parse_embeddings(response.json())
            all_vecs.extend(vecs)
        else:
            raise RuntimeError(f"Embedding call failed: {response.status_code} {response.text[:500]}")
    
    arr = np.array(all_vecs, dtype="float32")
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    arr = arr / norms
    return arr

print("Generating embeddings...")
embeddings = embed_databricks(chunk_texts).astype("float32")
print(f"Embedded {len(chunk_texts)} chunks")
print(f"Embedding dimension: {embeddings.shape[1]}")



## Save Chunks and Embeddings to Delta



In [ ]:

# Create DataFrame with chunks and embeddings
chunk_df = pd.DataFrame({
    "chunk_id": list(range(len(chunk_texts))),
    "reference_number": [m["reference_number"] for m in chunk_metadata],
    "section": [m["section"] for m in chunk_metadata],
    "chunk_index": [m["chunk_index"] for m in chunk_metadata],
    "chunk_text": chunk_texts,
    "embedding": [emb.tolist() for emb in embeddings]
})

print(f"Created DataFrame with {len(chunk_df)} rows")

# Convert to Spark DataFrame and save
print(f"Saving chunks to Delta table: {CHUNKS_TABLE}")
spark_chunk_df = spark.createDataFrame(chunk_df)
spark_chunk_df.write.format("delta").mode("overwrite").saveAsTable(CHUNKS_TABLE)

print("Chunks saved successfully")



## Create Document Lookup Table



In [ ]:

# Create a lookup table for full documents
doc_lookup_df = pd.DataFrame({
    "reference_number": all_ids,
    "full_document": docs
})

DOC_LOOKUP_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`complaint_documents`"

print(f"Saving document lookup to: {DOC_LOOKUP_TABLE}")
spark_doc_df = spark.createDataFrame(doc_lookup_df)
spark_doc_df.write.format("delta").mode("overwrite").saveAsTable(DOC_LOOKUP_TABLE)

print("Document lookup saved successfully")



## Save Pipeline Metadata



In [ ]:

# Save metadata about this pipeline run
metadata = {
    "pipeline_run_date": datetime.now().isoformat(),
    "num_complaints": len(all_ids),
    "num_chunks": len(chunk_texts),
    "embedding_dimension": int(embeddings.shape[1]),
    "embedding_endpoint": EMBEDDING_ENDPOINT,
    "chunk_size": 512,
    "chunk_overlap": 128,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "chunks_table": CHUNKS_TABLE,
    "documents_table": DOC_LOOKUP_TABLE,
    "vector_search_index": VECTOR_SEARCH_INDEX
}

metadata_df = pd.DataFrame([metadata])

print(f"Saving pipeline metadata to: {METADATA_TABLE}")
spark_metadata_df = spark.createDataFrame(metadata_df)
spark_metadata_df.write.format("delta").mode("overwrite").saveAsTable(METADATA_TABLE)

print("Pipeline metadata saved successfully")



## Verify Vector Search Index



In [ ]:

print(f"Vector Search Index: {VECTOR_SEARCH_INDEX}")
print()
print("Note: Ensure the Vector Search index is created and synced with the chunks table.")
print("The index should be configured to use the 'embedding' column.")
print()
print("To create/update the index, use the Databricks Vector Search UI or API.")



## Pipeline Summary



In [ ]:

print("="*80)
print("RAG PIPELINE SETUP COMPLETE")
print("="*80)
print()
print(f"Source complaints: {len(all_ids)}")
print(f"Total chunks created: {len(chunk_texts)}")
print(f"Average chunks per complaint: {len(chunk_texts) / len(docs):.1f}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print()
print("Tables created:")
print(f"  1. Chunks: {CHUNKS_TABLE}")
print(f"  2. Documents: {DOC_LOOKUP_TABLE}")
print(f"  3. Metadata: {METADATA_TABLE}")
print()
print("Next steps:")
print("  1. Verify Vector Search index is synced")
print("  2. Use the RAG Summarization notebook to generate summaries")
print()
print("="*80)



## Done

The RAG pipeline is now set up and persisted in Delta tables.

You can now use the summarization notebook to generate summaries for any complaint.

Re-run this notebook if:
- New complaints are added to the source table
- You want to change chunking parameters
- You want to regenerate embeddings

